## 注意
此 notebook 為選用步驟，作者已將處理完成的 `combined.pkl` 上傳至 Google Drive，
並在 `feature_engineering.ipynb` 中自動下載，無需實際執行。

In [ ]:
# 從 PhysioNet 下載 Challenge 2019 資料集
# 只下載 training_setA 和 training_setB
!wget -r -N -c -np https://physionet.org/files/challenge-2019/1.0.0/training/training_setA/
!wget -r -N -c -np https://physionet.org/files/challenge-2019/1.0.0/training/training_setB/

In [ ]:
import pandas as pd
import pdb
import numpy as np
import glob

In [ ]:
# 取得 training_setA 和 setB 所有 PSV 檔案
files1 = glob.glob('training_setA/*.psv')
files2 = glob.glob('training_setB/*.psv')
files = np.concatenate((files1, files2))  # 合併兩個資料夾的檔案

df_list = []
for ind, f in enumerate(files):
    # 從檔案路徑取得病人 ID，讀取 PSV 檔案並新增 patient 欄位
    patient_id = f.split('/')[1].split('.')[0] # 例如 ID = p000001）
    df = pd.read_csv(f, sep='|')
    df = df.assign(patient=patient_id)

    # 重新定義 SepsisLabel：將敗血症發生前六小時的標籤（1）改為 0
    # 讓模型學習敗血症發生當下，而非發生前六小時
    df.loc[df[df['SepsisLabel'] == 1].head(6).index.values, 'SepsisLabel'] = 0

    # 每處理 1000 筆印一次進度（共 40000 筆）
    if ind % 1000 == 0:
        print(ind)

    # 將當前病人資料加入清單
    df_list.append(df)

# 合併所有病人資料、重置索引，並儲存為 pickle 檔案
df = pd.concat(df_list)
df = df.reset_index(drop=True)
df.to_pickle('combined.pkl')
